# 01 Data Processing

Loads CDC PLACES (2021 release, 2019 data) and the USDA Food Access Research Atlas (2019), then merges and cleans them into `data/processed/tract_health_food_access_2019.csv`.

Install the package once from the repo root with `uv sync`

In [ ]:
# Reload the ac209a library automatically when its code changes (no kernel restart needed)
%load_ext autoreload
%autoreload 2

from ac209a import build_dataset as bd

### A. [CDC PLACES](https://catalog.data.gov/dataset/places-local-data-for-better-health-census-tract-data-2021-release)

**Dataset Description**: This dataset contains model-based census tract-level estimates for the PLACES 2021 release. PLACES is the expansion of the original 500 Cities project and covers the entire United States—50 states and the District of Columbia (DC)—at county, place, census tract, and ZIP Code Tabulation Area (ZCTA) levels. It represents a first-of-its kind effort to release information uniformly on this large scale for local areas at 4 geographic levels. Estimates were provided by the Centers for Disease Control and Prevention (CDC), Division of Population Health, Epidemiology and Surveillance Branch. PLACES was funded by the Robert Wood Johnson Foundation (RWJF) in conjunction with the CDC Foundation. The dataset includes estimates for 29 measures: 4 chronic disease-related health risk behaviors, 13 health outcomes, 3 health status, and 9 on use of preventive services. These estimates can be used to identify emerging health problems and to help develop and carry out effective, targeted public health prevention activities. Because the small area model cannot detect effects due to local interventions, users are cautioned against using these estimates for program or policy evaluations. Data sources used to generate these model-based estimates include Behavioral Risk Factor Surveillance System (BRFSS) 2019 or 2018 data, Census Bureau 2010 population data, and American Community Survey (ACS) 2015–019 or 2014–2018 estimates. The 2021 release uses 2019 BRFSS data for 22 measures and 2018 BRFSS data for 7 measures (all teeth lost, dental visits, mammograms, cervical cancer screening, colorectal cancer screening, core preventive services among older adults, and sleeping less than 7 hours a night). Seven measures are based on the 2018 BRFSS because the relevant questions are only asked every other year in the BRFSS. More information about the methodology can be found at www.cdc.gov/places.

In [2]:
places = bd.load_places()

print(places.shape)
places.head()

(140676, 9)


,Year,StateAbbr,GEOID,Data_Value_Unit,Data_Value,Low_Confidence_Limit,High_Confidence_Limit,Geolocation,MeasureId
9,2019,AL,01027959000,%,28.3,27.3,29.3,POINT (-85.87716485 33.30474877),DEPRESSION
35,2019,ME,23011010801,%,22.7,21.3,24.2,POINT (-69.8608252 44.33169058),DEPRESSION
111,2019,ME,23021960700,%,14.9,13.8,16.0,POINT (-69.20447634 45.18074924),MHLTH
148,2019,AL,01119011300,%,23.0,22.0,24.0,POINT (-88.13334736 32.59803651),DEPRESSION
178,2019,AK,02122001200,%,19.2,18.8,19.7,POINT (-151.268306 59.42412393),DEPRESSION


#### Quicklook

In [3]:
import plotly.express as px

# One measure at a time: keep the MHLTH rows (one row per tract)
mh = places[places["MeasureId"] == "MHLTH"].dropna(subset=["Data_Value"]).copy()

# Tract centroid coordinates from "POINT (lon lat)"
coords = mh["Geolocation"].str.extract(r"POINT \((?P<lon>\S+) (?P<lat>\S+)\)").astype(float)
mh["lon"] = coords["lon"]
mh["lat"] = coords["lat"]

# Sort so high-distress tracts are drawn on top
mh = mh.sort_values("Data_Value")

fig = px.scatter_map(
    mh, lat="lat", lon="lon",
    color="Data_Value", range_color=[12, 22], color_continuous_scale="Reds",
    hover_name="GEOID",
    hover_data={"StateAbbr": True, "Data_Value": True,
                "Low_Confidence_Limit": True, "High_Confidence_Limit": True,
                "lat": False, "lon": False},
    labels={"Data_Value": "Mental distress (%)"},
    zoom=3.5, center={"lat": 39, "lon": -96}, width=1000, height=700,
    map_style="carto-positron",
    title="Frequent mental distress by census tract, 2019 (CDC PLACES 2021 release)",
)
fig.update_traces(marker_size=3)
fig.show()

### B. [Food Access Research Atlas - Large Retailer Access Map](https://www.ers.usda.gov/data-products/food-access-research-atlas/documentation/large-retailer-access-map-reference-guide)


In [4]:
food = bd.load_food_access()
print(food.shape)   
food.head()

(72531, 20)


,GEOID,State,County,Urban,Pop2010,OHU2010,LILATracts_1And10,PovertyRate,MedianFamilyIncome,LA1and10,LAPOP1_10,lapophalfshare,lahunvhalfshare,lapop1share,lahunv1share,lapop10share,lahunv10share,TractSeniors,TractHUNV,TractSNAP
0,01001020100,Alabama,Autauga County,1,1912,693,0,11.336453,81250.0,1,1896.493338,100.000000,0.792272,99.188982,0.792272,NaN,NaN,221.0,6.0,102.0
1,01001020200,Alabama,Autauga County,1,2170,743,1,17.876788,49000.0,1,1260.991651,100.000000,12.472887,58.110214,9.004514,NaN,NaN,214.0,89.0,156.0
2,01001020300,Alabama,Autauga County,1,3373,1256,0,15.046030,62609.0,1,1551.703935,84.697873,3.086528,46.003674,0.000000,NaN,NaN,439.0,99.0,172.0
3,01001020400,Alabama,Autauga County,1,4386,1722,0,2.845210,70607.0,1,1363.466885,83.243525,1.127900,31.086796,0.461172,NaN,NaN,904.0,21.0,98.0
4,01001020500,Alabama,Autauga County,1,10766,4082,0,15.150891,96334.0,1,2643.095161,72.249639,4.014997,24.550392,1.352569,NaN,NaN,1126.0,230.0,339.0


#### Quicklook

In [5]:
import pandas as pd
import plotly.express as px

# Tract centroids from the Census Bureau's 2019 gazetteer (2010-based tract codes)
url = "https://www2.census.gov/geo/docs/maps-data/data/gazetteer/2019_Gazetteer/2019_Gaz_tracts_national.zip"
gaz = pd.read_csv(url, sep="\t", dtype={"GEOID": str})
gaz.columns = gaz.columns.str.strip()           # the last column name has trailing spaces

# Add lat/lon only; Food Access values are unchanged
food_map = food.merge(gaz[["GEOID", "INTPTLAT", "INTPTLONG"]], on="GEOID")

# Drop tracts with no value; sort so the worst-access tracts are drawn on top
food_map = food_map.dropna(subset=["lahunv1share"]).sort_values("lahunv1share")
print("Tracts plotted:", len(food_map))

fig = px.scatter_map(
    food_map, lat="INTPTLAT", lon="INTPTLONG",
    color="lahunv1share", range_color=[0, 15], color_continuous_scale="Oranges",
    hover_name="GEOID",
    hover_data={"State": True, "County": True, "Urban": True, "lahunv1share": True,
                "INTPTLAT": False, "INTPTLONG": False},
    labels={"lahunv1share": "No vehicle & >1 mi (%)"},
    zoom=3.5, center={"lat": 39, "lon": -96}, width=1000, height=700,
    map_style="carto-positron",
    title="Housing units with no vehicle and >1 mile from a supermarket (% of tract), 2019",
)
fig.update_traces(marker_size=3)
fig.show()


Tracts plotted: 52565


## Build the merged dataset

In [6]:
df = bd.build(places, food)
df.head()

Merged rows: 140672 | tracts: 70336
After cleaning: 140666 | tracts: 70333
Saved /Users/julianfontes/Library/Mobile Documents/com~apple~CloudDocs/School/Fall2026/DS1/project/ac209a/data/processed/tract_health_food_access_2019.csv


,GEOID,State,StateAbbr,County,Geolocation,Urban,MeasureId,Year,Data_Value,Data_Value_Unit,...,lahunvhalfshare,lahunv1share,lahunv10share,Pop2010,OHU2010,PovertyRate,MedianFamilyIncome,TractSeniors,TractHUNV,TractSNAP
801,01001020100,Alabama,AL,Autauga County,POINT (-86.49001486 32.47718251),1,DEPRESSION,2019,26.6,%,...,0.792272,0.792272,NaN,1912,693,11.336453,81250.0,221.0,6.0,102.0
384,01001020100,Alabama,AL,Autauga County,POINT (-86.49001486 32.47718251),1,MHLTH,2019,17.4,%,...,0.792272,0.792272,NaN,1912,693,11.336453,81250.0,221.0,6.0,102.0
1298,01001020200,Alabama,AL,Autauga County,POINT (-86.47338874 32.47425254),1,DEPRESSION,2019,23.9,%,...,12.472887,9.004514,NaN,2170,743,17.876788,49000.0,214.0,89.0,156.0
849,01001020200,Alabama,AL,Autauga County,POINT (-86.47338874 32.47425254),1,MHLTH,2019,18.5,%,...,12.472887,9.004514,NaN,2170,743,17.876788,49000.0,214.0,89.0,156.0
691,01001020300,Alabama,AL,Autauga County,POINT (-86.4602024 32.47543618),1,DEPRESSION,2019,26.7,%,...,3.086528,0.000000,NaN,3373,1256,15.046030,62609.0,439.0,99.0,172.0


In [7]:
import numpy as np
import xyzservices.providers as xyz
from bokeh.io import output_notebook, show
from bokeh.layouts import row
from bokeh.models import ColorBar, ColumnDataSource
from bokeh.palettes import Reds9, Oranges9
from bokeh.plotting import figure
from bokeh.transform import linear_cmap

output_notebook()

# One row per tract: keep the MHLTH rows (food access values are the same on every measure row)
tracts = df[df["MeasureId"] == "MHLTH"].copy()

# Tract centroids from PLACES "POINT (lon lat)", converted to Web Mercator for the map tiles
coords = tracts["Geolocation"].str.extract(r"POINT \((?P<lon>\S+) (?P<lat>\S+)\)").astype(float)
R = 6378137  # Earth radius (m)
tracts["x"] = np.radians(coords["lon"]) * R
tracts["y"] = np.log(np.tan(np.pi / 4 + np.radians(coords["lat"]) / 2)) * R


def tract_map(column, title, palette, low, high, x_range=None, y_range=None):
    """One dot per tract, colored by `column`. Pass another map's ranges to link zoom/pan."""
    d = tracts.dropna(subset=[column]).sort_values(column)      # highest values drawn on top
    p = figure(title=title, x_axis_type="mercator", y_axis_type="mercator", width=650, height=450,
               x_range=x_range or (-14_000_000, -7_400_000),   # continental U.S.
               y_range=y_range or (2_800_000, 6_400_000),
               tools="pan,wheel_zoom,box_zoom,reset,hover", active_scroll="wheel_zoom",
               tooltips=[("Tract", "@GEOID"), ("State", "@State"), (title, f"@{column}{{0.0}}")])
    p.add_tile(xyz.Esri.WorldGrayCanvas)                        # free light-gray basemap, no API key
    cmap = linear_cmap(column, palette[::-1], low, high)         # light = low, dark = high
    p.scatter("x", "y", source=ColumnDataSource(d[["x", "y", "GEOID", "State", column]]),
              size=3, color=cmap, line_color=None)
    p.add_layout(ColorBar(color_mapper=cmap["transform"], width=10), "right")
    p.axis.visible = False
    return p


mental = tract_map("Data_Value", "Frequent mental distress (%)", Reds9, 12, 22)
access = tract_map("lahunv1share", "No vehicle & >1 mi from supermarket (%)", Oranges9, 0, 15,
                   x_range=mental.x_range, y_range=mental.y_range)   # shared ranges = linked zoom

show(row(mental, access))

Loading BokehJS ...